## Replicacao dados de cancer de mama utilizando pseudo-valores para S(t)

Formula dos pseudo-valores para S(t):
$$Y_{ij} = n\hat{S}_{KM}(t_j) - (n-1)\hat{S}_{KM}^{(-i)}(t_j)$$

In [2]:
# Carregando bibliotecas

library(survival)
library(dplyr)
library(tidyr)

In [ ]:
dados_treino <- read.csv("../../../dados/dados-processados/dados_treino.csv", stringsAsFactors = FALSE)
dados_teste <- read.csv("../../../dados/dados-processados/dados_teste.csv", stringsAsFactors = FALSE)

covariaveis_rede <- setdiff(
  intersect(names(dados_treino), names(dados_teste)),
  c("id_paciente", "tempo", "delta_t")
)

prepara_base <- function(df, set_label, covariaveis) {
  base <- df %>%
    select(id_paciente, tempo, delta_t, all_of(covariaveis)) %>%
    mutate(
      ID = as.numeric(id_paciente),
      FAILTIME = as.numeric(tempo),
      FAILCENS = as.numeric(delta_t),
      SET = set_label
    ) %>%
    select(ID, FAILTIME, FAILCENS, all_of(covariaveis), SET) %>%
    filter(FAILTIME > 0, !is.na(FAILCENS))
  # Imputação de dados faltantes
  for (nm in covariaveis) {
    if (is.numeric(base[[nm]])) {
      med <- stats::median(base[[nm]], na.rm = TRUE)
      if (is.na(med)) med <- 0
      base[[nm]][is.na(base[[nm]])] <- med
    } else {
      base[[nm]] <- as.character(base[[nm]])
      base[[nm]][is.na(base[[nm]]) | base[[nm]] == ""] <- "NA"
    }
  }

  base
}

cancer_mama <- bind_rows(
  prepara_base(dados_treino, "train", covariaveis_rede),
  prepara_base(dados_teste, "test", covariaveis_rede)
) %>%
  arrange(ID)

n_distinct(cancer_mama$ID)
head(cancer_mama)

[1] 6076

,ID,FAILTIME,FAILCENS,rhc_raca_cor,instrucao,est_cong,rhc_historico_familiar_cancer,rhc_historico_alcool,rhc_historico_tabaco,rhc_origem_encamiamento,⋯,rhc_diagnostico_e_tratamentos_anteriores,rhc_idade_no_diagnostico_tumor,macroregiao,pndr_renda,faixa_etcat,ano,tipo_hcido,diff_data_1consulta,diff_data_tratamento,SET
,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<int>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<int>,<dbl>,<chr>,<int>,<int>,<chr>,<int>,<int>,<chr>
1,1,16,1,Branca,0,2,Não,NA,Nunca,Não SUS,⋯,Com diag./Sem trat.,73,3,Média Renda,5,2014,8500,12,42,train
2,2,60,1,Não Branca,0,2,Não,NA,Sim,SUS,⋯,Sem diag./Sem trat.,75,3,Média Renda,5,2012,8500,-104,69,train
3,3,1,1,Não Branca,0,2,NA,NA,NA,SUS,⋯,Sem diag./Sem trat.,93,1,Alta Renda,5,2018,8500,-41,0,test
4,4,13,1,Não Branca,0,1,Sim,NA,NA,SUS,⋯,Com diag./Sem trat.,29,2,Média Renda,1,2013,8500,19,33,test
5,5,69,1,Não Branca,1,2,NA,NA,NA,SUS,⋯,Com diag./Sem trat.,38,1,Baixa Renda,1,2010,8500,83,99,train
6,6,7,1,Não Branca,1,2,NA,NA,NA,SUS,⋯,Sem diag./Sem trat.,43,1,Alta Renda,2,2017,min100,-10,46,train


In [4]:
grade_tempos <- quantile(
  cancer_mama$FAILTIME[cancer_mama$FAILCENS > 0],
  probs = seq(0.05, 0.95, length.out = 10),
  names = FALSE
)

grade_tempos

[1]  2.0  8.0 12.0 16.0 21.1 28.0 34.0 42.0 53.0 78.0

In [5]:
get_sobrevivencia <- function(fit, tempos) {
  summary(fit, times = tempos, extend = TRUE)$surv
}

calcula_pseudo <- function(time, status, tempos) {
  n <- length(time)
  m <- length(tempos)

  fit_completo <- survfit(Surv(time, status) ~ 1)
  s_completo <- get_sobrevivencia(fit_completo, tempos)

  pseudo <- matrix(NA, nrow = n, ncol = m)

  for (i in seq_len(n)) {
    fit_sem_i <- survfit(Surv(time[-i], status[-i]) ~ 1)
    s_sem_i <- get_sobrevivencia(fit_sem_i, tempos)
    pseudo[i, ] <- n * s_completo - (n - 1) * s_sem_i
  }

  colnames(pseudo) <- paste0("t_", round(tempos, 6))
  pseudo
}

pseudo_valores <- calcula_pseudo(
  time = cancer_mama$FAILTIME,
  status = cancer_mama$FAILCENS,
  tempos = grade_tempos
)

n <- nrow(cancer_mama)
m <- length(grade_tempos)

base_expandida <- cancer_mama[rep(seq_len(n), each = m), c("ID", covariaveis_rede, "SET"), drop = FALSE]

entrada_rede <- base_expandida %>%
  mutate(
    TIME = rep(grade_tempos, times = n),
    PSEUDO = as.vector(t(pseudo_valores))
  ) %>%
  select(ID, all_of(covariaveis_rede), TIME, PSEUDO, SET)

head(entrada_rede, 20)

,ID,rhc_raca_cor,instrucao,est_cong,rhc_historico_familiar_cancer,rhc_historico_alcool,rhc_historico_tabaco,rhc_origem_encamiamento,rhc_estadiamento_clinico,rhc_primeiro_tratamento_recebido_no_hospital_nenhum,⋯,macroregiao,pndr_renda,faixa_etcat,ano,tipo_hcido,diff_data_1consulta,diff_data_tratamento,TIME,PSEUDO,SET
,<dbl>,<chr>,<dbl>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<dbl>,<chr>,<int>,<int>,<chr>,<int>,<int>,<dbl>,<dbl>,<chr>
1,1,Branca,0,2,Não,NA,Nunca,Não SUS,NA,Não,⋯,3,Média Renda,5,2014,8500,12,42,2.0,1.00000538,train
1.1,1,Branca,0,2,Não,NA,Nunca,Não SUS,NA,Não,⋯,3,Média Renda,5,2014,8500,12,42,8.0,1.00013470,train
1.2,1,Branca,0,2,Não,NA,Nunca,Não SUS,NA,Não,⋯,3,Média Renda,5,2014,8500,12,42,12.0,1.00036735,train
1.3,1,Branca,0,2,Não,NA,Nunca,Não SUS,NA,Não,⋯,3,Média Renda,5,2014,8500,12,42,16.0,-0.03903342,train
1.4,1,Branca,0,2,Não,NA,Nunca,Não SUS,NA,Não,⋯,3,Média Renda,5,2014,8500,12,42,21.1,-0.03859288,train
1.5,1,Branca,0,2,Não,NA,Nunca,Não SUS,NA,Não,⋯,3,Média Renda,5,2014,8500,12,42,28.0,-0.03803687,train
1.6,1,Branca,0,2,Não,NA,Nunca,Não SUS,NA,Não,⋯,3,Média Renda,5,2014,8500,12,42,34.0,-0.03742833,train
1.7,1,Branca,0,2,Não,NA,Nunca,Não SUS,NA,Não,⋯,3,Média Renda,5,2014,8500,12,42,42.0,-0.03683670,train
1.8,1,Branca,0,2,Não,NA,Nunca,Não SUS,NA,Não,⋯,3,Média Renda,5,2014,8500,12,42,53.0,-0.03610448,train


In [6]:
dir_entrada <- file.path("..", "dados", "entrada")
dir.create(dir_entrada, recursive = TRUE, showWarnings = FALSE)

write.csv(entrada_rede, file.path(dir_entrada, "entrada_rede.csv"), row.names = FALSE)
write.csv(cancer_mama, file.path(dir_entrada, "cancer_mama_bruto.csv"), row.names = FALSE)

list.files(dir_entrada)

[1] "cancer_mama_bruto.csv" "entrada_rede.csv"